# BirdCLEF+ 2026 — Submission Notebook

Generic inference notebook. Works with **any** `.keras` model whose input shape is `(128, 311, 1)` (the mel-spectrogram of a 5-second window at 32 kHz) and whose output is 234 sigmoid probabilities.

**To submit a different model:** change the `MODEL_PATH` constant in the next cell. Everything else stays the same.

**Before running on Kaggle:**
1. Upload your `.keras` file as a Kaggle Dataset (e.g. named `birdclef-2026-my-model`).
2. Attach that dataset to this notebook (Add data → your dataset).
3. Attach the **BirdCLEF+ 2026** competition data (Add data → BirdCLEF+ 2026).
4. Update `MODEL_PATH` below to point at your uploaded weights.
5. Set the notebook accelerator to **None** (CPU only — required for this competition).
6. Run all cells. `submission.csv` will appear in the notebook output.

**Inference budget:** 90 minutes CPU-only for ~700 one-minute soundscapes. The pipeline below does 12 windows per soundscape (one per 5-second segment), so ~8,400 model forward passes total. At ~45 ms/window on a typical CPU model that's ~6 minutes — well within budget.

In [41]:
# ─────────────────────────────────────────────────────────────────────
# CONFIGURATION — edit MODEL_PATH to submit a different model
# ─────────────────────────────────────────────────────────────────────

# Path to your model weights inside the Kaggle environment.
# If you uploaded the .keras file as a Kaggle Dataset named e.g. 'birdclef-my-model',
# Kaggle mounts it at /kaggle/input/birdclef-my-model/. Update accordingly.
MODEL_PATH = "/kaggle/input/datasets/antniofaustino2004/birdclef-2026-regular-exp8-clean/model_clean.keras"

# Standard Kaggle paths. Don't change these unless the competition layout differs.
TEST_SOUNDSCAPE_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
SAMPLE_SUB_CSV       = "/kaggle/input/competitions/birdclef-2026/sample_submission.csv"
OUTPUT_CSV           = "submission.csv"

# Audio / mel parameters. MUST match what was used during training
# (these are the values from baseline.py — do not change unless you
# retrained the model with different ones).
SAMPLE_RATE   = 32_000
DURATION_SEC  = 5.0      # one prediction window
WINDOWS_PER_SOUNDSCAPE = 12   # 60s / 5s = 12
N_FFT         = 1024
HOP_LENGTH    = 512
N_MELS        = 128

In [42]:
# ─────────────────────────────────────────────────────────────────────
# Imports
# ─────────────────────────────────────────────────────────────────────
import os
# Keras 3 backend — Kaggle's image has tensorflow available; use it.
# If your model was saved with a torch backend the file is still loadable
# from tensorflow, but if you hit a deserialization issue, switch this
# to 'torch' (pip install torch keras inside the notebook if needed).
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import keras

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

Keras version: 3.10.0
Keras backend: tensorflow


In [43]:
# ─────────────────────────────────────────────────────────────────────
# Label space — read from sample_submission.csv so we match exactly
# ─────────────────────────────────────────────────────────────────────
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
LABELS = [c for c in sample_sub.columns if c != "row_id"]
NUM_CLASSES = len(LABELS)
print(f"Label space: {NUM_CLASSES} classes (from sample_submission.csv)")
assert NUM_CLASSES == 234, f"Expected 234 classes, got {NUM_CLASSES}"

Label space: 234 classes (from sample_submission.csv)


In [44]:
# ─────────────────────────────────────────────────────────────────────
# Mel-spectrogram pipeline
# Lifted verbatim from baseline.py — keeps train/test consistent.
# Vectorised: builds all frames at once via stride tricks + batched FFT.
# ─────────────────────────────────────────────────────────────────────
def _build_mel_filterbank(sr: int, n_fft: int, n_mels: int) -> np.ndarray:
    def hz_to_mel(hz): return 2595.0 * np.log10(1.0 + hz / 700.0)
    def mel_to_hz(mel): return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)
    f_min, f_max = 20.0, sr / 2.0
    mel_pts = np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2)
    hz_pts  = mel_to_hz(mel_pts)
    bins    = np.floor((n_fft + 1) * hz_pts / sr).astype(int)
    n_freqs = n_fft // 2 + 1
    fb = np.zeros((n_mels, n_freqs), dtype=np.float32)
    for i in range(1, n_mels + 1):
        l, c, r = bins[i - 1], bins[i], bins[i + 1]
        if c > l: fb[i - 1, l:c] = np.linspace(0, 1, c - l, dtype=np.float32)
        if r > c: fb[i - 1, c:r] = np.linspace(1, 0, r - c, dtype=np.float32)
    return fb


MEL_FB   = _build_mel_filterbank(SAMPLE_RATE, N_FFT, N_MELS)
HANN_WIN = np.hanning(N_FFT).astype(np.float32)


def waveform_to_mel(audio: np.ndarray) -> np.ndarray:
    """Log-mel spectrogram of a 1D waveform. Returns (n_mels, time_frames)."""
    n = len(audio)
    n_frames = max(1, 1 + (n - N_FFT) // HOP_LENGTH)
    needed = (n_frames - 1) * HOP_LENGTH + N_FFT
    if n < needed:
        audio = np.pad(audio, (0, needed - n))
    frames = np.lib.stride_tricks.as_strided(
        audio,
        shape=(n_frames, N_FFT),
        strides=(audio.strides[0] * HOP_LENGTH, audio.strides[0]),
        writeable=False,
    )
    windowed = frames * HANN_WIN
    spec = np.abs(np.fft.rfft(windowed, axis=1)) ** 2
    spec = spec.T
    mel = MEL_FB @ spec
    mel = np.log1p(mel)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.astype(np.float32)

In [45]:
# ─────────────────────────────────────────────────────────────────────
# Compatibility resave: rebuild the architecture and load weights only
# ─────────────────────────────────────────────────────────────────────
# The .keras file in the uploaded dataset was saved by Keras 3.14.1,
# which emits BatchNormalization config with `renorm`/`renorm_clipping`/
# `renorm_momentum` fields. Kaggle's newer Keras (≥3.15?) removed those
# args and refuses to deserialize the saved config. The architecture
# itself is unchanged — only the JSON config has stale fields.
#
# Workaround: rebuild the architecture in code (clean BN layers, no stale
# fields), then load_weights() reads only the H5 weights chunk inside the
# .keras zip and ignores the config. Save fresh.

from keras import layers as _kl

NUM_CLASSES_RESAVE = 234
INPUT_SHAPE_RESAVE = (128, 311, 1)


def _build_crnn_lstm_clean():
    """Same architecture as compare_models.py's build_crnn_lstm, but built
    fresh by this Keras version so no deprecated args end up in the config."""
    inputs = keras.Input(shape=INPUT_SHAPE_RESAVE)

    x = _kl.Conv2D(32, 3, padding="same", use_bias=False)(inputs)
    x = _kl.BatchNormalization()(x)
    x = _kl.ReLU()(x)
    x = _kl.MaxPooling2D(pool_size=(2, 2))(x)

    x = _kl.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = _kl.BatchNormalization()(x)
    x = _kl.ReLU()(x)
    x = _kl.MaxPooling2D(pool_size=(2, 2))(x)

    x = _kl.Conv2D(128, 3, padding="same", use_bias=False)(x)
    x = _kl.BatchNormalization()(x)
    x = _kl.ReLU()(x)

    x = _kl.Permute((2, 1, 3))(x)
    x = _kl.Reshape((x.shape[1], x.shape[2] * x.shape[3]))(x)

    x = _kl.Bidirectional(_kl.LSTM(64, return_sequences=False))(x)
    x = _kl.Dropout(0.3)(x)
    outputs = _kl.Dense(NUM_CLASSES_RESAVE, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="crnn_lstm")


print("Rebuilding architecture in current Keras...")
rebuilt = _build_crnn_lstm_clean()
print(f"  rebuilt parameters: {rebuilt.count_params():,}")

print(f"Loading weights from {MODEL_PATH}...")
# load_weights reads the H5 weights chunk and ignores the JSON config,
# bypassing the renorm field problem entirely.
rebuilt.load_weights(MODEL_PATH)
print("  weights loaded")

CLEAN_PATH = "/kaggle/working/model_clean.keras"
rebuilt.save(CLEAN_PATH)
print(f"  saved clean model to {CLEAN_PATH}")

# Re-point MODEL_PATH at the clean file for the rest of the notebook
MODEL_PATH = CLEAN_PATH

Rebuilding architecture in current Keras...
  rebuilt parameters: 2,253,962
Loading weights from /kaggle/input/datasets/antniofaustino2004/birdclef-2026-regular-exp8-clean/model_clean.keras...
  weights loaded
  saved clean model to /kaggle/working/model_clean.keras


In [46]:
# ─────────────────────────────────────────────────────────────────────
# Load the model
# ─────────────────────────────────────────────────────────────────────
assert Path(MODEL_PATH).exists(), (
    f"Model file not found at {MODEL_PATH}. "
    f"Did you attach your dataset to the notebook?"
)
print(f"Loading model from {MODEL_PATH} ...")
t0 = time.perf_counter()
# compile=False skips deserializing the training-time loss
# (e.g. WeightedBCE / FocalLoss generated by the agents) —
# we only need the model for inference here.
model = keras.models.load_model(MODEL_PATH, compile=False)
print(f"  loaded in {time.perf_counter() - t0:.1f}s")
print(f"  input shape : {model.input_shape}")
print(f"  output shape: {model.output_shape}")
print(f"  parameters  : {model.count_params():,}")

# Sanity checks — fail loudly here rather than 45 min into inference.
assert model.output_shape[-1] == NUM_CLASSES, (
    f"Model output ({model.output_shape[-1]}) doesn't match label "
    f"space ({NUM_CLASSES})."
)
assert model.input_shape[1] == N_MELS, (
    f"Model input n_mels ({model.input_shape[1]}) doesn't match "
    f"pipeline N_MELS ({N_MELS}). The model was trained with "
    f"different parameters than this notebook uses."
)

Loading model from /kaggle/working/model_clean.keras ...
  loaded in 0.2s
  input shape : (None, 128, 311, 1)
  output shape: (None, 234)
  parameters  : 2,253,962


In [47]:
# ─────────────────────────────────────────────────────────────────────
# Inference helpers
# ─────────────────────────────────────────────────────────────────────
def split_soundscape_into_windows(audio: np.ndarray) -> np.ndarray:
    """Slice a 60s mono waveform into 12 fixed 5s windows.
    Pads with zeros if the file is shorter than 60s."""
    target_per_window = int(SAMPLE_RATE * DURATION_SEC)
    target_total      = target_per_window * WINDOWS_PER_SOUNDSCAPE
    if len(audio) < target_total:
        audio = np.pad(audio, (0, target_total - len(audio)))
    else:
        audio = audio[:target_total]
    return audio.reshape(WINDOWS_PER_SOUNDSCAPE, target_per_window)


def read_soundscape(path: Path) -> np.ndarray:
    """Read mono audio at the project sample rate."""
    audio, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if audio.ndim == 2:                     # stereo -> mono
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        raise ValueError(
            f"Sample rate mismatch in {path}: {sr} != {SAMPLE_RATE}. "
            f"This pipeline expects 32 kHz."
        )
    return audio


def predict_soundscape(path: Path) -> np.ndarray:
    """Returns (12, NUM_CLASSES) — one row per 5s window."""
    audio = read_soundscape(path)
    windows = split_soundscape_into_windows(audio)
    # Build the (12, 128, 311, 1) batch and predict in one call —
    # batching all 12 windows together is much faster than 12 separate
    # forward passes (one model call vs 12 on CPU is ~5x faster).
    mels = np.stack([waveform_to_mel(w) for w in windows], axis=0)
    mels = mels[..., np.newaxis]
    preds = model.predict(mels, verbose=0)
    return preds

In [48]:
# ─────────────────────────────────────────────────────────────────────
# Run inference on all test soundscapes
# ─────────────────────────────────────────────────────────────────────
test_dir = Path(TEST_SOUNDSCAPE_DIR)
soundscape_files = sorted(test_dir.glob("*.ogg"))
if not soundscape_files:
    soundscape_files = sorted(test_dir.glob("*.wav"))
print(f"Found {len(soundscape_files)} test soundscape files")

rows = []
t_inference_start = time.perf_counter()
for i, ss_path in enumerate(soundscape_files):
    preds = predict_soundscape(ss_path)              # (12, 234)
    # row_id format: <filename_without_extension>_<end_seconds>
    # end_seconds: 5, 10, 15, ..., 60 (windows 1..12)
    stem = ss_path.stem
    for w, p in enumerate(preds):
        end_s = int((w + 1) * DURATION_SEC)
        row = {"row_id": f"{stem}_{end_s}"}
        # Raw sigmoid output, no post-processing
        for j, lab in enumerate(LABELS):
            row[lab] = float(p[j])
        rows.append(row)

    # Progress + ETA every 25 files
    if (i + 1) % 25 == 0 or (i + 1) == len(soundscape_files):
        elapsed = time.perf_counter() - t_inference_start
        rate    = (i + 1) / elapsed                  # files / sec
        eta_s   = (len(soundscape_files) - (i + 1)) / max(rate, 1e-9)
        print(f"  [{i + 1:>4}/{len(soundscape_files)}] "
              f"elapsed={elapsed:6.1f}s  "
              f"rate={rate:5.2f} files/s  "
              f"eta={eta_s/60:5.1f} min")

total_inference_min = (time.perf_counter() - t_inference_start) / 60
print(f"\nInference complete: {len(rows)} rows in {total_inference_min:.1f} min")
if total_inference_min > 85:
    print("  [!] WARNING: inference time is close to the 90-minute budget.")
    print("      Consider a smaller model or quicker mel pipeline for headroom.")

Found 0 test soundscape files

Inference complete: 0 rows in 0.0 min


In [49]:
# ─────────────────────────────────────────────────────────────────────
# Build the submission DataFrame and verify it
# ─────────────────────────────────────────────────────────────────────
expected_cols = sample_sub.columns.tolist()

if not rows:
    # Dev mode: test_soundscapes is empty (hidden until submission).
    # Write an empty submission with the correct header so Kaggle
    # accepts the notebook and shows the "Submit to Competition" button.
    print("[!] No test soundscapes were processed (expected during dev — "
          "Kaggle reveals the real test set only at submission time).")
    sub = pd.DataFrame(columns=expected_cols)
else:
    sub = pd.DataFrame(rows)
    missing = set(expected_cols) - set(sub.columns)
    extra   = set(sub.columns)  - set(expected_cols)
    assert not missing, f"missing columns in submission: {sorted(missing)[:10]}"
    assert not extra,   f"extra columns in submission:   {sorted(extra)[:10]}"
    sub = sub[expected_cols]

    # Sanity checks only when we actually have data
    assert sub["row_id"].is_unique, "row_id values are not unique"
    prob_cols = [c for c in sub.columns if c != "row_id"]
    vals = sub[prob_cols].to_numpy()
    assert np.isfinite(vals).all(), "submission contains NaN or inf values"
    assert (vals >= 0).all() and (vals <= 1).all(), (
        f"probabilities outside [0, 1]: range=[{vals.min()}, {vals.max()}]"
    )
    print(f"Probability range: [{vals.min():.4f}, {vals.max():.4f}]")
    print(f"Mean probability:  {vals.mean():.4f}")

print(f"Submission shape: {sub.shape}")
print(f"First rows:")
print(sub.head(3))

[!] No test soundscapes were processed (expected during dev — Kaggle reveals the real test set only at submission time).
Submission shape: (0, 235)
First rows:
Empty DataFrame
Columns: [row_id, 1161364, 116570, 1176823, 1491113, 1595929, 209233, 22930, 22956, 22961, 22967, 22973, 22983, 22985, 23150, 23154, 23158, 23176, 23724, 24279, 24285, 24287, 24321, 244024, 25073, 25092, 25214, 326272, 41970, 43435, 47144, 47158son01, 47158son02, 47158son03, 47158son04, 47158son05, 47158son06, 47158son07, 47158son08, 47158son09, 47158son10, 47158son11, 47158son12, 47158son13, 47158son14, 47158son15, 47158son16, 47158son17, 47158son18, 47158son19, 47158son20, 47158son21, 47158son22, 47158son23, 47158son24, 47158son25, 476521, 516975, 517063, 555123, 555145, 555146, 64898, 65377, 65380, 66971, 67107, 67252, 70711, 738183, 74113, 74580, 760266, ashgre1, astcra1, bafcur1, baffal1, banana, barant1, batbel1, baymac, bbwduc, bcwfin2, bkcdon, bkhpar, blchaw1, blheag1, blttit1, bncfly, bobfly1, brcmar1, b

In [50]:
# ─────────────────────────────────────────────────────────────────────
# Write submission.csv
# ─────────────────────────────────────────────────────────────────────
sub.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {OUTPUT_CSV} ({Path(OUTPUT_CSV).stat().st_size / 1024:.1f} KB)")

Wrote submission.csv (1.8 KB)
